# Lab2.4 — API PYNQ vs AxiGPIO vs MMIO

În acest task comparăm trei niveluri de acces la același hardware: API-ul de nivel înalt al Base Overlay-ului, driverul `AxiGPIO` și accesul direct cu `MMIO`.


## Obiective

- compararea nivelurilor de abstractizare;
- verificarea faptului că toate metodele controlează același IP hardware;
- măsurarea costului software al operațiilor de scriere;
- diferențierea dintre overhead software și latența hardware propriu-zisă.


## 1. Cele trei niveluri

```text
Nivel înalt        base.leds[0].on()
                        ↓
Driver PYNQ        AxiGPIO.write()
                        ↓
Acces registre     MMIO.write()
                        ↓
                   AXI-Lite
                        ↓
                    AXI GPIO
                        ↓
                       LED
```


## 2. Inițializare


In [ ]:
from pynq.overlays.base import BaseOverlay
from pynq.lib import AxiGPIO
from pynq import MMIO
import time

base = BaseOverlay("base.bit")

def find_led_gpio():
    for name, info in base.ip_dict.items():
        text = (name + " " + info.get("type", "")).lower()
        if "gpio" in text and "led" in name.lower() and "rgb" not in name.lower():
            return name
    raise RuntimeError("LED GPIO not found")

led_name = find_led_gpio()
led_info = base.ip_dict[led_name]

axi_leds = AxiGPIO(led_info).channel1
axi_leds.setdirection("out")
axi_leds.setlength(4)

mmio_leds = MMIO(led_info["phys_addr"], led_info["addr_range"])
mmio_leds.write(0x04, 0x00000000)

print("LED GPIO:", led_name)

## 3. Același efect prin trei metode

Executați pe rând următoarele celule și verificați LED0.


In [ ]:
# Nivel înalt
base.leds[0].on()
time.sleep(0.5)
base.leds[0].off()

In [ ]:
# Driver AxiGPIO
axi_leds.write(0x1, 0xF)
time.sleep(0.5)
axi_leds.write(0x0, 0xF)

In [ ]:
# MMIO direct
mmio_leds.write(0x00, 0x1)
time.sleep(0.5)
mmio_leds.write(0x00, 0x0)

## 4. Măsurarea overhead-ului software

Măsurăm timpul pentru un număr mare de perechi ON/OFF. Nu introducem `sleep()` și nu afișăm nimic în interiorul buclei.

> **Important:** acesta nu este timpul de propagare electrică până la LED. Măsurăm în principal costul apelurilor Python și al accesului software la periferic.


In [ ]:
N = 1000

def benchmark_high_level(n=N):
    led = base.leds[0]
    t0 = time.perf_counter()
    for _ in range(n):
        led.on()
        led.off()
    return time.perf_counter() - t0

def benchmark_axi_gpio(n=N):
    t0 = time.perf_counter()
    for _ in range(n):
        axi_leds.write(1, 0xF)
        axi_leds.write(0, 0xF)
    return time.perf_counter() - t0

def benchmark_mmio(n=N):
    t0 = time.perf_counter()
    for _ in range(n):
        mmio_leds.write(0x00, 1)
        mmio_leds.write(0x00, 0)
    return time.perf_counter() - t0

t_high = benchmark_high_level()
t_axi  = benchmark_axi_gpio()
t_mmio = benchmark_mmio()

print(f"High-level API : {t_high*1000:.3f} ms")
print(f"AxiGPIO        : {t_axi*1000:.3f} ms")
print(f"MMIO           : {t_mmio*1000:.3f} ms")

Calculăm și timpul mediu pe o operație de scriere. Fiecare iterație conține două scrieri.


In [ ]:
writes = 2 * N

print(f"High-level API : {1e6*t_high/writes:.3f} us/write")
print(f"AxiGPIO        : {1e6*t_axi/writes:.3f} us/write")
print(f"MMIO           : {1e6*t_mmio/writes:.3f} us/write")

## 5. Interpretarea rezultatelor

Nu presupuneți că metoda cu cel mai mic timp într-o singură rulare este automat «mai bună». Repetați măsurarea de cel puțin cinci ori și observați variațiile introduse de Python și Linux.

În aplicațiile Edge, alegerea nivelului de acces depinde de compromis:

- **API de nivel înalt** — cod simplu și lizibil;
- **AxiGPIO** — control explicit al perifericului;
- **MMIO** — control direct asupra registrelor și o imagine clară a interfeței hardware.


## 6. Exercițiu

Repetați benchmark-ul de **5 ori** pentru fiecare metodă. Calculați media și deviația standard a timpului pe scriere. Prezentați rezultatele într-un tabel.


In [ ]:
# Scrieți aici soluția


## 7. Întrebări

1. De ce timpii măsurați variază între rulări?
2. Ce parte din lanț măsurăm cu `time.perf_counter()` în acest experiment?
3. De ce aceste rezultate nu reprezintă latența hardware pură a AXI GPIO?
4. În ce situație ați prefera MMIO în locul API-ului de nivel înalt?
